In [1]:
"""
Main training loop.
Handles environment interaction, agent learning,
logging, and checkpoint saving.
"""

import numpy as np
from config import Config
from env import make_env
from agent import DoubleDQNAgent


def train():
    """
    Run the full training loop:
      1. Create environment and agent.
      2. For each step:
         a. Select action (epsilon-greedy).
         b. Step environment.
         c. Store transition.
         d. Call agent.learn() if enough samples.
         e. Sync target network periodically.
         f. Log episode rewards.
         g. Save checkpoints.
    """
    config = Config()

    # ----- setup -----
    env, preprocessor, stacker = make_env(
        config.ENV_NAME, config.FRAME_SIZE, config.FRAME_STACK
    )
    n_actions = env.action_space.n
    device = "cuda" if __import__("torch").cuda.is_available() else "cpu"

    agent = DoubleDQNAgent(
        n_frames=config.FRAME_STACK,
        n_actions=n_actions,
        config=config,
        device=device,
    )

    # ----- metrics -----
    episode_rewards = []
    episode_reward = 0.0
    episode_count = 0
    losses = []

    # ----- initial reset -----
    obs, info = env.reset()
    state = stacker.reset(preprocessor.process(obs))
    # ----- training loop -----
    for step in range(1, config.TOTAL_STEPS + 1):
    
        # Select action
        action = agent.select_action(state)
    
        # Environment step
        next_obs, reward, terminated, truncated, info = env.step(action)
    
        done = terminated or truncated
    
        # Preprocess next observation
        processed_next = preprocessor.process(next_obs)
    
        # Update frame stack
        next_state = stacker.append(processed_next)
    
        # Store transition
        agent.store_transition(
            state,
            action,
            reward,
            next_state,
            done
        )
    
        # Learn
        loss = agent.learn()
    
        if loss is not None:
            losses.append(loss)
    
        # Update episode reward
        episode_reward += reward
    
        # Move to next state
        state = next_state
    
        # Episode finished
        if done:
    
            episode_rewards.append(episode_reward)
            episode_count += 1
    
            if episode_count % 10 == 0:
    
                avg_reward = np.mean(
                    episode_rewards[-10:]
                )
    
                avg_loss = (
                    np.mean(losses[-100:])
                    if len(losses) > 0
                    else 0.0
                )
    
                print(
                    f"Episode {episode_count} | "
                    f"Step {step} | "
                    f"Avg Reward: {avg_reward:.2f} | "
                    f"Avg Loss: {avg_loss:.4f}"
                )
    
            # Reset environment
            obs, info = env.reset()
    
            state = stacker.reset(
                preprocessor.process(obs)
            )
    
            episode_reward = 0.0
    
        # Save checkpoint
        if step % config.SAVE_INTERVAL == 0:
    
            checkpoint_path = (
                f"checkpoint_{step}.pth"
            )
    
            agent.save(checkpoint_path)
    
            print(
                f"Checkpoint saved: "
                f"{checkpoint_path}"
            )

In [2]:
if __name__ == "__main__":
    rewards = train()

Episode 10 | Step 17860 | Avg Reward: -0.80 | Avg Loss: 0.1014
Episode 20 | Step 35720 | Avg Reward: 0.60 | Avg Loss: 0.0843
Checkpoint saved: checkpoint_50000.pth
Episode 30 | Step 53580 | Avg Reward: 2.00 | Avg Loss: 0.0785
Episode 40 | Step 71440 | Avg Reward: -3.90 | Avg Loss: 0.0616
Episode 50 | Step 89300 | Avg Reward: -14.70 | Avg Loss: 0.0478
Checkpoint saved: checkpoint_100000.pth
Episode 60 | Step 107160 | Avg Reward: -30.40 | Avg Loss: 0.0385
Episode 70 | Step 125020 | Avg Reward: -29.40 | Avg Loss: 0.0330
Episode 80 | Step 142880 | Avg Reward: -34.60 | Avg Loss: 0.0264
Checkpoint saved: checkpoint_150000.pth
Episode 90 | Step 160740 | Avg Reward: -18.90 | Avg Loss: 0.0178
Episode 100 | Step 178600 | Avg Reward: -22.90 | Avg Loss: 0.0123
Episode 110 | Step 196460 | Avg Reward: -15.80 | Avg Loss: 0.0101
Checkpoint saved: checkpoint_200000.pth
Episode 120 | Step 214320 | Avg Reward: -16.50 | Avg Loss: 0.0096
Episode 130 | Step 232180 | Avg Reward: -14.50 | Avg Loss: 0.0074
Che